In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import *
from src.common.spark_utils import standardize_columns
from src.common.quality_events import log_pipeline_failure, quarantine_rejected_rows, reconcile_merge_target
from delta.tables import DeltaTable

In [ ]:
# Configure the Bronze root for this workspace or job.
dbutils.widgets.text("bronze_root", "abfss://bronze@ubereaststorage.dfs.core.windows.net", "Bronze storage root")
BRONZE_ROOT = dbutils.widgets.get("bronze_root").rstrip("/")


In [0]:
# Read the Bronze source and log source-read failures before re-raising.
try:
    df_promotions = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{BRONZE_ROOT}/csv/promotions/")
    )
except Exception as exc:
    log_pipeline_failure(spark, "promotions", "source_read", exc)
    raise
source_rows = df_promotions.count()
print(f"promotions.source_rows={source_rows}")
display(df_promotions)
df_promotions.printSchema()

In [0]:
# standardize the columns

# Standardize columns and capture invalid rows before the existing dedup/filter steps.
df_promotions = standardize_columns(df_promotions)

duplicate_promotion_keys = (
    df_promotions
    .filter(col("promo_id").isNotNull() & col("restaurant_id").isNotNull())
    .groupBy("promo_id", "restaurant_id").count().filter(col("count") > 1).count()
)
duplicate_promotion_key_values = (
    df_promotions
    .filter(col("promo_id").isNotNull() & col("restaurant_id").isNotNull())
    .groupBy("promo_id", "restaurant_id").count().filter(col("count") > 1)
    .select("promo_id", "restaurant_id")
)
duplicate_promotion_rows = df_promotions.join(
    duplicate_promotion_key_values, on=["promo_id", "restaurant_id"], how="inner"
)
quarantine_rejected_rows(
    spark, duplicate_promotion_rows, "promotions",
    "duplicate_promo_restaurant_key_group_no_winner_policy",
)
if duplicate_promotion_keys:
    raise ValueError(
        f"Promotions Silver input contains {duplicate_promotion_keys} duplicate composite-key groups; "
        "all rows in those groups were quarantined and publishing was stopped"
    )

null_promo_ids = df_promotions.filter(col("promo_id").isNull()).count()
null_restaurant_ids = df_promotions.filter(col("restaurant_id").isNull()).count()
quarantine_run_id = str(__import__("uuid").uuid4())
quarantine_rejected_rows(
    spark,
    df_promotions.filter(col("promo_id").isNull() | col("restaurant_id").isNull()),
    "promotions", "missing_promo_id_or_restaurant_id", run_id=quarantine_run_id,
)
if null_promo_ids or null_restaurant_ids:
    raise ValueError(
        "Promotions Silver input contains rows with null merge keys; "
        "records were quarantined and publishing was stopped"
    )
quarantine_rejected_rows(
    spark,
    df_promotions.filter(col("discount_value").isNull() | (col("discount_value") <= 0)),
    "promotions", "discount_value_null_or_nonpositive", run_id=quarantine_run_id,
)

# Existing business transformations retained below.


                       
#deduplication 
df_promotions = df_promotions.drop_duplicates(["promo_id","restaurant_id"]) 

#applying business rules
df_promotions = df_promotions.filter(col("discount_value")>0)

display(df_promotions)










In [0]:
try:
    table_name = "ubereats_databricks1.silver.promotions"
    
    if not spark.catalog.tableExists(table_name):
    
        df_promotions.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)
    
    else:
    
        target = DeltaTable.forName(spark, table_name)
    
        (
            target.alias("t")
            .merge(
                df_promotions.alias("s"),
                "t.promo_id = s.promo_id AND t.restaurant_id = s.restaurant_id",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
except Exception as exc:
    # Record publish failures without replacing the original exception.
    failure_record = [{
        "run_id": str(__import__("uuid").uuid4()),
        "dataset_name": "promotions",
        "stage": "silver_publish",
        "error_type": type(exc).__name__,
        "error_message": str(exc)[:2000],
        "recorded_at_utc": str(spark.sql("SELECT current_timestamp() AS ts").first()["ts"]),
    }]
    try:
        spark.createDataFrame(failure_record).write.format("delta").mode("append").saveAsTable(
            "ubereats_databricks1.silver.pipeline_failure_events"
        )
    except Exception as log_exc:
        print("promotions.failure_log_error=" + str(log_exc)[:500])
    raise

# Persist a compact successful-run quality record for later monitoring.
quality_table = "ubereats_databricks1.silver.pipeline_quality_results"
quality_record = [{
    "run_id": str(__import__("uuid").uuid4()),
    "dataset_name": "promotions",
    "source_rows": int(source_rows),
    "prepared_rows": int(df_promotions.count()),
    "target_rows_after": int(spark.table(table_name).count()),
    "status": "SUCCEEDED",
    "recorded_at_utc": str(spark.sql("SELECT current_timestamp() AS ts").first()["ts"]),
}]
spark.createDataFrame(quality_record).write.format("delta").mode("append").saveAsTable(quality_table)
print("promotions.quality.persisted_run_id=" + quality_record[0]["run_id"])
